<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%202/2_st445_week2_lab2_webscraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 2 seminar, lab 2: web scraping


In the API lab, the data came in a form designed for programs. A web page is designed for people, so you have to find the data inside it, and then **clean** it: what you scrape is almost never ready to analyse. In this lab you:

* scrape **this course's own guide** on the LSE website, and clean its text into tidy fields;
* scrape **every course listed in the MSc Data Science programme regulations**, and clean the messy text into a table of course codes, titles and units;
* practise on sites built for scraping, many items and many pages at a time;
* study a real case where researchers scraped and published 70,000 dating-site profiles, and write the collection plan that should have existed.

### How the lab is organised

| Part | What happens | Time |
|---|---|---|
| **A. Guided demo** | Your tutor runs two examples with you: the ST445 course guide, then the MSc Data Science course list. Each has a *scrape* stage and a *clean* stage. | 25 min |
| **B. Pause and analyse** | Short exercises: change a section, handle missing elements, clean text, find the bug in last year's code, scrape a practice bookshop. | 25 min |
| **C. Research exercises** | The dating-site case study, a pre-collection check for your project, and a reproducible scrape. | 10 min in the seminar, then after |

Cell marks: 🟢 run and read · 🟡 fill the gaps marked `____` · 🔵 write the code yourself · ✍️ write a short answer. Boxes titled **Method card** explain each new function and the minimum you need to set.

**Before you start:** **File → Save a copy in Drive**, and rename it `<yourname>_week2_scraping_lab.ipynb`.

## Set-up

`requests` downloads pages, `BeautifulSoup` reads HTML, `re` finds patterns in text, and `pandas` makes tables. `RobotFileParser` reads a site's robots.txt file, and `urljoin` turns a link such as `page-2.html` into a full address.

In [ ]:
# 🟢 Run this first
import json, re, time, hashlib
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
from bs4 import BeautifulSoup

USER_AGENT = "ST445 seminar lab (London School of Economics; teaching use)"
HEADERS = {"User-Agent": USER_AGENT}

def check(condition, hint="Not yet. Read the task again."):
    print("✅ Correct" if condition else f"❌ {hint}")

def filled(*values):
    """False (with a reminder) if any value still contains ____."""
    if any("____" in str(v) for v in values):
        print("✋ Replace every ____ first, then run this cell again.")
        return False
    return True

print("Ready.")

---
# Part A. Guided demo

## A1. The professional routine for scraping

With an API you can start by sending a request. With scraping, the first step happens **before** any download, because you are taking data from a page that was not designed to be collected.

| Step | What you do | Why |
|---|---|---|
| 1 | **Check whether you may scrape:** is there an API or a download? What do robots.txt and the terms of use say? Is there personal data? | Scraping is the last resort, and many sites forbid it |
| 2 | **Request the page** with a `User-Agent` that says who you are, and a timeout | The site knows who is asking; your notebook never hangs |
| 3 | **Check the status code** | 200 means you have the page; anything else means stop |
| 4 | **Check you received HTML**, and set the text encoding | An error page or a wrong encoding breaks everything after |
| 5 | **Parse, and check the element you need exists** before extracting | Websites change their layout; fail loudly rather than collect nothing quietly |
| 6 | **Extract, then clean**, save the raw HTML and a provenance record, and pause before the next request | Raw text is never analysis-ready; the raw copy lets you re-clean without downloading again |

---
# Example 1: the ST445 course guide

Open the page in a browser tab now, then right-click the course title and choose **Inspect**.

In [ ]:
# 🟢 The page
COURSE_URL = "https://www.lse.ac.uk/resources/calendar/courseGuides/ST/2026_ST445.htm"
parts = urlparse(COURSE_URL)
print("Site:", parts.netloc, "| Path:", parts.path)

## A2. Step 1: check robots.txt

> **Method card: `RobotFileParser`**
>
> * `rp = RobotFileParser(robots_url)` creates a checker for one site's robots.txt (**required**: the full robots.txt URL).
> * `rp.read()` downloads and reads the file.
> * `rp.can_fetch(user_agent, page_url)` returns `True` if that user agent may fetch that page, otherwise `False`.

In [ ]:
# 🟢 What does robots.txt say?
def robots_allows(url, user_agent=USER_AGENT):
    """True or False from the site's robots.txt, or None if it cannot be read."""
    p = urlparse(url)
    rp = RobotFileParser(f"{p.scheme}://{p.netloc}/robots.txt")
    try:
        rp.read()
        return rp.can_fetch(user_agent, url)
    except Exception as err:
        print("Could not read robots.txt:", type(err).__name__)
        return None

robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"
try:
    print(requests.get(robots_url, headers=HEADERS, timeout=20).text[:500])
except requests.RequestException as err:
    print("Could not download robots.txt:", err)

allowed = robots_allows(COURSE_URL)
print("\nrobots.txt allows our user agent to fetch the course guide:", allowed)

**What you should see:** the LSE's robots.txt contains `Disallow: /`. The `/` means **the whole site**, so automated programs are asked not to fetch any LSE page, including this one, and `can_fetch` returns `False`. **This page should not be scraped.**

We scrape it here **as an exception, for learning purposes only**: two pages, one request each, with an honest `User-Agent`. In your project, a `False` here means you do not scrape the site: find another source, or ask the site owner for permission. The robots.txt result goes into the provenance record below.

## A3. Steps 2 to 4: request the page and check what came back

`fetch_html` runs steps 2 to 4 and keeps a copy of each page it downloads in `cache_html/`. If a request fails, it uses the cached copy, or a copy of the page you saved from your browser and uploaded (cell below).

In [ ]:
# 🟢 Steps 2 to 4 as a function
CACHE = Path("cache_html"); CACHE.mkdir(exist_ok=True)

def cache_path(url):
    return CACHE / (re.sub(r"[^A-Za-z0-9]+", "_", url).strip("_")[-120:] + ".html")

def fetch_html(url, allowed_by_robots=True, learning_exception=False, pause=1.0):
    """Steps 2 to 4. Returns (html_text, source), where source is 'live', 'cache' or None."""
    path = cache_path(url)
    if allowed_by_robots is False and not learning_exception:
        print("Not fetching: robots.txt does not allow this page.")
        return None, None
    try:
        print("Requesting:", url)
        r = requests.get(url, headers=HEADERS, timeout=30)                 # step 2
        print("Status code:", r.status_code)                               # step 3
        content_type = r.headers.get("Content-Type", "")
        print("Content type:", content_type)                               # step 4
        if r.status_code == 200 and "html" in content_type.lower():
            r.encoding = r.apparent_encoding or "utf-8"                    # step 4: encoding
            path.write_text(r.text, encoding="utf-8")
            time.sleep(pause)                                              # be polite
            return r.text, "live"
        print("Not a usable HTML page.")
    except requests.RequestException as err:
        print("Request failed:", type(err).__name__)
    if path.exists():
        print("Using the cached copy.")
        return path.read_text(encoding="utf-8"), "cache"
    return None, None

**If the request fails in Colab**, open the page in your browser, choose **File → Save page as… → Web page, HTML only**, run the cell below, and upload the file. It is stored as the cached copy of that URL.

In [ ]:
# 🟢 Optional: upload a page you saved from your browser
def upload_saved_page(url):
    from google.colab import files
    for name, data in files.upload().items():
        cache_path(url).write_text(data.decode("cp1252", errors="replace"), encoding="utf-8")
        print("Saved", name, "as the cached copy of", url)

# upload_saved_page(COURSE_URL)        # remove the # to run

In [ ]:
# 🟢 Steps 2 to 4 for the course guide (an exception to robots.txt, for learning purposes)
html, source = fetch_html(COURSE_URL, allowed_by_robots=allowed, learning_exception=True)
print("\nUsing:", source, "|", len(html), "characters of HTML")

**What you should see:** status code **200**, content type `text/html`, and `Using: live`.

> **Why set the encoding?** HTML arrives as bytes. This page declares the `windows-1252` character set; if the bytes are read with the wrong one, `O’Reilly` turns into `Oâ€™Reilly` or an invisible control character. `r.apparent_encoding` looks at the bytes to choose.

## A4. Step 5: parse the page, and look before you extract

> **Method card: `BeautifulSoup(markup, parser)`**
>
> * `markup` (**required**): the HTML text.
> * `parser`: use `"html.parser"`, which comes with Python.
>
> It returns a `soup`: the whole page as a tree you can search.

> **Method card: finding elements**
>
> | Method | Returns | Example |
> |---|---|---|
> | `soup.find(tag, id=..., class_=...)` | the **first** match, or `None` | `soup.find("span", class_="course-code")` |
> | `soup.find_all(tag, id=...)` | a **list** of all matches (empty if none) | `soup.find_all(id="assessment-Content")` |
> | `soup.select_one(css)` | the first match for a CSS selector, or `None` | `soup.select_one("h2 span.course-title")` |
> | `soup.select(css)` | a list of all matches for a CSS selector | `soup.select('[id$="-Label"]')` |
>
> For one element `el`: `el.get_text()` gives its visible text; `el.get("href")` reads an **attribute** (or `None`); `el["id"]` reads one that must exist.

In [ ]:
# 🟢 Parse, then look at the course heading
soup = BeautifulSoup(html, "html.parser")
print("Page title:", soup.title.get_text(strip=True))
heading = soup.find("h2")
print(heading.prettify())

**What you should see:** the `<h2>` holds three `<span>` elements with **class names** that say what they contain: `course-code`, `course-value` and `course-title`. Class names like these are the most reliable handles on a page.

In [ ]:
# 🟢 Step 5: confirm the structure we rely on is there, then extract by class
code_el = soup.find("span", class_="course-code")
if code_el is None:
    raise ValueError("No span.course-code: the layout has changed, so stop and inspect the page again.")
course_code = code_el.get_text(strip=True)
course_value = soup.find("span", class_="course-value").get_text(strip=True)
course_title = soup.find("span", class_="course-title").get_text(strip=True)
print(course_code, "|", course_value, "|", course_title)

Now the sections. Inspect *Course convenor* in the browser: each section is a **label** paragraph with an `id` ending in `-Label` (for example `courseConvenorResponsible-Label`), followed by a **content** `<div>` whose `id` ends in `-Content`. List every label on the page with a CSS selector: `[id$="-Label"]` means "any element whose `id` ends with `-Label`".

In [ ]:
# 🟢 Look first: every section label, and its id
for label in soup.select('[id$="-Label"]'):
    print(f'{label["id"]:<35} {label.get_text(" ", strip=True)}')

**What you should see:** labels such as *Course convenor*, *Availability*, *Requisites*, *Course content*, *Teaching*, *Formative assessment*, *Indicative reading*, *Assessment* and *Key facts*. The part of the `id` before `-Label` is the key to the matching content: `courseConvenorResponsible-Label` goes with `courseConvenorResponsible-Content`.

## A5. Step 6, first half: extract the raw text

The functions below find a section by its label and return its content **raw**, exactly as the page has it, so we can see what cleaning is needed. One detail matters: on this page the *Assessment* content appears in **three** `<div>` elements that share the same `id` (strictly, invalid HTML, but common), so we use `find_all`, not `find`.

In [ ]:
# 🟢 Raw content of a section, found by its label
def section_divs(soup, label):
    """All content elements for the section whose label starts with label (not case-sensitive)."""
    for lab in soup.select('[id$="-Label"]'):
        if lab.get_text(" ", strip=True).lower().startswith(label.lower()):
            key = lab["id"][: -len("-Label")]
            return soup.find_all(id=f"{key}-Content") or soup.find_all(id=f"{key}-content")
    return []

def raw_section(soup, label):
    """The raw, uncleaned text of a section, or None if the label is not on the page."""
    divs = section_divs(soup, label)
    return "".join(d.get_text() for d in divs) if divs else None

print(len(section_divs(soup, "Assessment")), "Assessment divs")
raw_assessment = raw_section(soup, "Assessment")
print(raw_assessment)

It looks almost fine when printed. Now print it with `repr`, which shows the characters that are really there:

In [ ]:
# 🟢 What is really in the text?
print(repr(raw_assessment))

**What you should see:** line breaks (`\n`), runs of spaces, and `\xa0`. That last one is a **non-breaking space**, written `&nbsp;` in HTML: it looks like a space on screen but is a different character, so `"30% in total)\xa0In addition"` will not split or match the way you expect. These characters are invisible on the page and break comparisons, splitting and matching. That is why cleaning comes next.

## A6. Step 6, second half: clean, one step at a time

### Cleaning step 1: normalise the white space

Replace non-breaking spaces with ordinary spaces, turn every run of white space into one space, and trim the ends.

> **Method card: `re.sub(pattern, replacement, text)`** replaces every match of a **regular expression** `pattern` in `text`. In a pattern, `\s` means any white-space character and `+` means "one or more", so `\s+` matches any run of white space. Write patterns as raw strings, `r"..."`, so Python leaves the backslashes alone.

In [ ]:
# 🟢 Cleaning step 1
def clean_text(text):
    """Normalise white space: non-breaking spaces become spaces; runs of white space become one space."""
    if text is None:
        return None
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def section_text(soup, label):
    """Cleaned text of a section, or None if the label is not on the page."""
    return clean_text(raw_section(soup, label))

assessment = clean_text(raw_assessment)
print(repr(assessment))

### Cleaning step 2: extract the parts you need with a pattern

Assessed components appear as *a name*, then *a percentage in brackets*: `Project (70%)`. A regular expression can capture both parts each time they appear.

> **Method card: regular expressions for extraction**
>
> | Piece | Means |
> |---|---|
> | `\d` | a digit; `\d+` one or more digits |
> | `\(` and `\)` | literal brackets (plain `(` and `)` have a special meaning) |
> | `( ... )` | a **group**: the part you want to keep |
> | `[A-Za-z][A-Za-z -]*?` | a letter, then as few letters, spaces or hyphens as possible |
>
> `re.findall(pattern, text)` returns every match; with two groups, each match is a pair `(group1, group2)`.

In [ ]:
# 🟢 Cleaning step 2: component and weight
pattern = r"([A-Za-z][A-Za-z -]*?)\s*\((\d+)%\)"
matches = re.findall(pattern, assessment)
components = pd.DataFrame(matches, columns=["component", "weight_pct"])
components["weight_pct"] = components["weight_pct"].astype(int)     # text to numbers
components

### Cleaning step 3: check the cleaned data makes sense

Cleaning can go wrong silently. A **validation check** compares the result with something you know must be true: the weights of the assessed components must add up to 100.

In [ ]:
# 🟢 Cleaning step 3: validate
print("Weights add up to", components["weight_pct"].sum())
check(components["weight_pct"].sum() == 100, "The weights do not add up to 100: look at the rows above.")

**This check fails on the real page, and that is the point.** The table above has an extra row: the explanatory paragraph mentions *take-home exam (70%)*, which describes the project a second time. The pattern did exactly what we asked, on text that says the same thing twice.

The fix comes from looking at the structure again: the **first** two Assessment `<div>` elements each hold one short line naming one component (`Project (70%).` and `Problem sets (30%).`); the third holds the explanation. So apply the pattern to each `<div>` separately, and keep only divs whose text is a single short line.

In [ ]:
# 🟢 Cleaning step 3, fixed: one component per short div
rows = []
for div in section_divs(soup, "Assessment"):
    text = clean_text(div.get_text())
    m = re.fullmatch(r"([A-Za-z][A-Za-z -]*?)\s*\((\d+)%\)\.?", text)     # the whole div must be "Name (NN%)."
    if m:
        rows.append({"component": m.group(1), "weight_pct": int(m.group(2))})
components = pd.DataFrame(rows)
print(components)
check(components["weight_pct"].sum() == 100, "The weights should add up to 100.")

> **Method card: `re.fullmatch(pattern, text)`** succeeds only if the **whole** text matches the pattern, unlike `re.search`, which finds a match anywhere. Use it when you know exactly what a clean value looks like.

### Cleaning step 4: "Label: value" lines into a dictionary

The *Key facts* box is a series of lines such as `Department: Statistics` and `Total students 2025/26: 113`. Inspect it: the last line, *Controlled access*, is not even inside a `<p>`. So take the text **line by line** (`get_text("\n")` puts a line break between elements), split each line at the first colon, and convert numbers.

In [ ]:
# 🟢 Cleaning step 4: key facts
key_facts_div = section_divs(soup, "Key facts")[0]
facts = {}
for line in key_facts_div.get_text("\n").split("\n"):
    line = clean_text(line)
    if ":" in line:
        name, value = line.split(":", 1)                  # split at the first colon only
        value = value.strip()
        facts[name.strip()] = int(value) if value.isdigit() else value
facts

**What you should see:** a dictionary such as `{'Department': 'Statistics', 'Course study period': 'Autumn Term', 'Unit value': 'Half unit', 'FHEQ level': 'Level 7', 'Total students 2025/26': 113, 'Average class size 2025/26': 28, 'Controlled access 2025/26': 'Yes'}`, with the two counts as **integers**.

### Cleaning step 5: a list into a clean table

The reading list is a `<ul>` of `<li>` items. Clean each item, then extract two things from the text: the **year** (four digits) and any **web address**.

In [ ]:
# 🟢 Cleaning step 5: the reading list
reading = []
for div in section_divs(soup, "Indicative reading"):
    for li in div.find_all("li"):
        item = clean_text(li.get_text()).replace("\u2019", "'").replace("\x92", "'")   # curly or mis-decoded apostrophes
        url = re.search(r"https?://\S+", item)
        year = re.search(r"\b(19|20)\d{2}\b", item)
        reading.append({"item": item, "type": "web" if url else "book",
                        "url": url.group(0) if url else None,
                        "year": int(year.group(0)) if year else None})
reading = pd.DataFrame(reading)
print(reading["type"].value_counts(), "\n")
reading

**What you should see:** 10 items: 7 books with a publication year and 3 web resources (Matplotlib, Seaborn and scikit-learn) with an address and no year.

### One tidy record

Write the sentence first: **one row is one course.** Then gather the cleaned fields.

In [ ]:
# 🟢 One row = one course, from cleaned fields
course = {
    "code": course_code,
    "title": course_title,
    "unit_value": course_value,
    "convenor": section_text(soup, "Course convenor"),
    "department": facts.get("Department"),
    "students_2025_26": facts.get("Total students 2025/26"),
    "project_weight_pct": int(components.loc[components["component"].str.contains("project", case=False), "weight_pct"].sum()),
    "n_reading_items": len(reading),
    "url": COURSE_URL,
}
pd.DataFrame([course]).T

### Save the raw page and a provenance record

The raw HTML is kept **exactly as received**, separately from what you cleaned, so you can re-clean later without downloading again. The record includes the robots.txt result.

In [ ]:
# 🟢 Raw HTML and provenance
RAW = Path("data/raw/html"); RAW.mkdir(parents=True, exist_ok=True)

def save_raw(html, url, name, source, robots, note=""):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")
    raw_file = RAW / f"{name}_{stamp}.html"
    raw_file.write_text(html, encoding="utf-8")
    record = {"collected_at": stamp, "url": url, "source": source, "user_agent": USER_AGENT,
              "robots_txt_allows": robots, "note": note, "file": raw_file.name,
              "sha256": hashlib.sha256(html.encode()).hexdigest()}
    with open("data/provenance.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record

print(json.dumps(save_raw(html, COURSE_URL, "st445_course_guide", source, allowed,
                          note="robots.txt disallows; scraped as an exception for learning purposes"), indent=2))

---
# Example 2: every course in the MSc Data Science regulations

The programme regulations list the courses students take, organised by **paper** (a slot in the programme). Open the page in a browser and **Inspect** one course: the list is a single HTML `<table>`.

In [ ]:
# 🟢 The page: robots.txt first, then steps 2 to 4 (an exception, for learning purposes)
PROGRAMME_URL = "https://www.lse.ac.uk/resources/calendar/programmeRegulations/taughtMasters/2026/MScDataScience.htm"
programme_allowed = robots_allows(PROGRAMME_URL)
print("robots.txt allows:", programme_allowed)

prog_html, prog_source = fetch_html(PROGRAMME_URL, allowed_by_robots=programme_allowed, learning_exception=True)
prog = BeautifulSoup(prog_html, "html.parser")
print("\nUsing:", prog_source)

## A7. Look before you extract

How many tables and rows are there, and what does a row look like **raw**? Print every row's cells with `repr`.

In [ ]:
# 🟢 Look first
table = prog.find("table")
rows = table.find_all("tr")
print(len(prog.find_all("table")), "table,", len(rows), "rows\n")
for tr in rows[:16]:
    print([repr(td.get_text()) for td in tr.find_all(["td", "th"])])

**What you should see** (33 rows in total):

```
["' '", "'\\nCourse code, title (unit value) '"]                   header
["'\\nPaper\\xa01 '", "'\\n \\nST445 Managing and Visualising Data (0.5) # \\n'"]
["''", "' '"]                                                        an empty spacer row
["'\\nPaper\\xa04 '", "'\\nCourses to the value of 1.5 unit(s), including at least 0.5 unit(s) of ST courses from the following:  '"]
["''", "'MY461 Social Network Analysis (0.5) (suspended 2026/27) '"]
```

Every kind of mess is here: line breaks and spaces around the text; a non-breaking space in `Paper\xa01`; empty spacer rows; a rule (*Courses to the value of 1.5 units...*) rather than a course; a `#` symbol after some courses; and a note *(suspended 2026/27)* after others. The paper number appears only on the **first** row of each paper.

So a course is not "the second cell of each row". It is **a pattern inside the cleaned text**: a course code, a title, a unit value in brackets, then perhaps a symbol and a note.

## A8. Extract and clean, one step at a time

### Cleaning step 1: each row's text, normalised, with its paper

The first cell holds the paper (`Paper 1`, or empty); the second holds the course or the rule. Where the first cell is empty, the row belongs to the paper above it, so we **carry the paper forward**.

In [ ]:
# 🟢 Cleaning step 1
row_texts, current_paper = [], None
for tr in rows:
    cells = tr.find_all("td")
    if len(cells) < 2:                              # header rows use <th>; skip rows without two cells
        continue
    paper = clean_text(cells[0].get_text())
    text = clean_text(cells[1].get_text())
    if paper:                                       # 'Paper 4' starts a new paper ...
        current_paper = paper
    if text:                                        # ... and empty spacer rows are dropped
        row_texts.append({"paper": current_paper, "text": text})
row_texts = pd.DataFrame(row_texts)
row_texts.head(8)

### Cleaning step 2: find course codes with a pattern

LSE course codes are two capital letters followed by three characters, the first of which is a digit: `ST445`, `MY459` (and some, such as `MG4F5`, have a letter in them). As a pattern: `[A-Z]{2}\d[A-Z0-9]{2}`.

> **Method card: more pattern pieces**
>
> | Piece | Means |
> |---|---|
> | `[A-Z]{2}` | exactly two capital letters |
> | `[A-Z0-9]{2}` | two characters, each a capital letter or a digit |
> | `\b` | a word boundary, so `XST4456` does not count |
> | `(?P<name> ... )` | a **named group**, read back with `match.group("name")` |
> | `(?: ... )?` | an **optional** part that is not kept as a group of its own |
> | `[^)]*` | any characters except a closing bracket |
>
> `re.search(pattern, text)` returns the first match or `None`; `re.finditer(pattern, text)` goes through **every** match.

In [ ]:
# 🟢 Cleaning step 2: try the pattern on the messiest rows first
COURSE = re.compile(r"\b(?P<code>[A-Z]{2}\d[A-Z0-9]{2})\b\s*(?P<title>.+?)\s*\((?P<units>\d+(?:\.\d+)?)\)"
                    r"\s*(?P<flag>#?)\s*(?:\((?P<note>[^)]*)\))?")

for example in ["ST445 Managing and Visualising Data (0.5) #",
                "MY461 Social Network Analysis (0.5) (suspended 2026/27)",
                "ST454 Bayesian Data Analysis (0.5) # (suspended 2026/27)",
                "Courses to the value of 1.5 unit(s), including at least 0.5 unit(s) of ST courses from the following:"]:
    m = COURSE.search(example)
    print(m.groupdict() if m else f"No course in: {example[:50]}...")

**What you should see:** the three course rows split into `code`, `title`, `units`, `flag` and `note` (the note is `None` where there is none), and **no match** for the rule row. Read the pattern left to right: a code; the shortest title followed by a unit value in brackets; an optional `#`; an optional note in brackets.

### Cleaning step 3: apply the pattern to every row

In [ ]:
# 🟢 Cleaning step 3
records = []
for _, row in row_texts.iterrows():
    for m in COURSE.finditer(row["text"]):
        records.append({"paper": row["paper"], **m.groupdict()})
courses = pd.DataFrame(records)
print(len(courses), "courses found;", len(row_texts) - len(courses), "rows were not courses:")
print(row_texts.loc[~row_texts["text"].apply(lambda t: bool(COURSE.search(t))), "text"].tolist())
courses.head(10)

### Cleaning step 4: fix the types, and turn symbols and notes into columns

* `units` is text: make it a number.
* `flag` is `#` or empty. The page explains `#` in its footnotes: read the footnote from the page rather than assuming, then make a True/False column.
* `note` is free text such as `suspended 2026/27`: make a True/False `suspended` column.

In [ ]:
# 🟢 Cleaning step 4
footnote = next((clean_text(p.get_text()) for p in prog.find_all("p")
                 if clean_text(p.get_text()).startswith("#")), None)
print("Footnote on the page:", footnote, "\n")

courses["units"] = courses["units"].astype(float)
courses["may_have_prerequisites"] = courses["flag"] == "#"
courses["suspended"] = courses["note"].fillna("").str.contains("suspended", case=False)
courses = courses.drop(columns=["flag"])
courses[courses["suspended"]]

**What you should see:** the footnote `# means there may be prerequisites for this course...`, and two suspended courses: MY461 *Social Network Analysis* and ST454 *Bayesian Data Analysis*.

### Cleaning step 5: validate against the page itself

On this page each course code is a **link** to its course guide. That gives an independent count to check against, and the guide address, which is better than building it ourselves.

In [ ]:
# 🟢 Cleaning step 5: validate, then add the guide address and department
links = {a.get_text(strip=True): a["href"] for a in table.find_all("a", href=True)
         if re.fullmatch(r"[A-Z]{2}\d[A-Z0-9]{2}", a.get_text(strip=True))}
print(len(links), "course links on the page;", len(courses), "courses parsed")

check(len(courses) == len(links), "Every course link should give one parsed course.")
check(set(courses["code"]) == set(links), "The parsed codes should be exactly the linked codes.")
check(courses["code"].is_unique, "One row should be one course.")

courses["guide_url"] = courses["code"].map(links)
courses["department"] = courses["code"].str[:2]
courses[["paper", "code", "title", "units", "may_have_prerequisites", "suspended", "department"]]

**What you should see:** every check ✅, with **23 courses** in 5 papers: three compulsory half-units (ST445, ST447, ST443), 19 options in Paper 4, and the ST498 Capstone Project (1.0) in Paper 5.

> **Method card: `series.map(dictionary)`** replaces each value with the dictionary's value for it: here, each course code with its guide address.

### Save the raw page, the clean table, and a provenance record

In [ ]:
# 🟢 Save
save_raw(prog_html, PROGRAMME_URL, "msc_data_science_regulations", prog_source, programme_allowed,
         note="robots.txt disallows; scraped as an exception for learning purposes")
Path("data/processed").mkdir(parents=True, exist_ok=True)
courses.to_csv("data/processed/msc_data_science_courses.csv", index=False)
print("Saved", len(courses), "courses to data/processed/msc_data_science_courses.csv")

In this examplke above you did: check robots.txt, request and check, look before you extract, then clean in small steps, each one checked: normalise white space, extract with patterns, validate (and fix what validation finds), fix types, and cross-check against the page. Part B asks you to do these steps yourself.

---
# Part B. Pause and analyse

The exercises use two sites built for scraping practice: **books.toscrape.com** and **quotes.toscrape.com**. Their purpose is to be scraped, so everyone can run the code at once. Open each one in a browser tab when it first appears.

## B1. Change the label

🟡 Use `section_text` to get two other sections of the course guide. Pick labels from the list of section labels printed in A4.

In [ ]:
label_1 = "____"
label_2 = "____"

if filled(label_1, label_2):
    for label in [label_1, label_2]:
        print(f"{label}:\n  {section_text(soup, label)}\n")
    check(section_text(soup, label_1) is not None and section_text(soup, label_2) is not None,
          "Use labels exactly as they start in the label list from A4.")

✍️ **Predict, then try.** What will `section_text(soup, "Exam timetable")` return, and why does it not crash?

*Your answer:*

## B2. When an element is missing

`find` and `select_one` return `None` when nothing matches. Calling `.get_text()` on `None` crashes the whole scrape. Run this to see the error you will meet most often in scraping:

In [ ]:
# 🟢 The most common scraping error
try:
    soup.find("h5").get_text()
except AttributeError as err:
    print("AttributeError:", err)

🟡 Complete two small helpers that return `None` instead of crashing. You will use them for the rest of the lab.

In [ ]:
def text_or_none(node, css):
    """Text of the first element matching css inside node, or None if there is none."""
    found = node.select_one(css)
    return found.get_text(" ", strip=True) if found ____ None else None

def attr_or_none(node, css, attribute):
    """An attribute of the first element matching css, or None."""
    found = node.select_one(css)
    return found.get(____) if found is not None else None

In [ ]:
# Test for B2
check(text_or_none(soup, "h5") is None and text_or_none(soup, "span.course-title") == course_title, "Return None when found is None.")
check(attr_or_none(soup, "a", "href") is not None and attr_or_none(soup, "h5", "href") is None, "Use found.get(attribute).")

## B3. Cleaning practice

### (a) Normalise, then compare

🟡 Two strings scraped from different pages look the same but are not equal. Complete `normalise` so they become equal: replace the non-breaking space, collapse white space, trim, and make the text lower case.

In [ ]:
a = "Problem\xa0set 1  (15%)\n"
b = "  problem set 1 (15%)"
print(repr(a), "==", repr(b), "->", a == b)

def normalise(text):
    text = text.replace("____", " ")
    text = re.sub(r"____", " ", text)
    return text.strip().____()

if filled(normalise.__code__.co_consts):
    print(repr(normalise(a)), "==", repr(normalise(b)), "->", normalise(a) == normalise(b))
    check(normalise(a) == normalise(b) == "problem set 1 (15%)", "Replace \\xa0, use \\s+, then lower().")

### (b) Write a pattern

🟡 Complete the pattern so it extracts the **unit value** (the number in brackets) from each string. Remember that brackets must be written `\(` and `\)`, and that the number may have a decimal point.

In [ ]:
samples = ["ST447 Data Analysis and Statistical Methods (0.5) #", "ST498 Capstone Project (1.0)", "MG4F5 Analytics for Business (0.5)"]
unit_pattern = r"____"

if filled(unit_pattern):
    units = [float(re.search(unit_pattern, s).group(1)) for s in samples]
    print(units)
    check(units == [0.5, 1.0, 0.5], r"Try \((\d+(?:\.\d+)?)\): a bracket, a group of digits with an optional decimal part, a bracket.")

### (c) Clean a new section of the course guide

The *Availability* section begins: *"This course is compulsory on the MSc in Data Science and MSc in Health Data Science. This course is available on ..."*

🔵 Using `section_text` and a pattern, extract the programmes on which ST445 is **compulsory** into a Python list `compulsory_on`, one programme per item. Hints: `re.search(r"compulsory on the (.+?)\.", text)` captures everything up to the first full stop; then split the captured text on `" and "` (and on `", "` if there are more than two).

In [ ]:
# Your code here: compulsory_on = [...]

In [ ]:
# Test for B3 (c)
check(compulsory_on == ["MSc in Data Science", "MSc in Health Data Science"],
      "Capture up to the first full stop, then split on ' and '.")

### (d) How much of the programme can you choose?

🔵 Using `courses` from Example 2, compute the **total unit value** of courses listed under each paper (`groupby("paper")["units"].sum()`), leaving out suspended courses. Then extract from `row_texts` the number of units a student must take in Paper 4 (the rule row says *Courses to the value of 1.5 unit(s)...*).

✍️ In two sentences: why is Paper 4's total so much larger than 1.5? What does that tell you about using scraped regulations as data without reading the text around them?

*Your answer:*

In [ ]:
# Your code here

## B4. What went wrong in last year's code?

This is the code last year's seminar used for the same programme page. Run it and look carefully at the output.

In [ ]:
# 🟢 Last year's code, unchanged except for the year and the fallback
old_soup = prog
courses_old, index_old = [], []
for i in range(len(old_soup.find_all("tr")) - 3):
    try:
        tr = old_soup.find_all("tr")[i + 3].find_all("td")[1].text
    except IndexError:
        continue
    new = tr[:5] + "," + tr[6:-2]
    if "4" in new:
        row = new.split(",")
        index_old.append(row[0])
        courses_old.append(row[1])
pd.DataFrame({"Course code": index_old, "Course name": courses_old}).head(10)

✍️ **Find at least four problems.** For each, say which line causes it and which cleaning step from Example 2 fixes it. Things to look for:

* Why do some codes look like `'\n \nST'` or `'ST   47'`? (What is in the first five characters of the raw text?)
* What does `tr[6:-2]` assume about every row, and when is it wrong?
* Why skip the first three rows, `[i + 3]`? What happens if the page adds a row at the top?
* What does `if "4" in new` keep and drop? (Think of a course code with no 4 in it, and a note that contains a 4.)
* What happens to a cell that lists two courses?
* Why does the code call `find_all("tr")` inside the loop, and what does that cost?

*Your answer:*

✍️ **Principle.** In one sentence: why is *"find a pattern in the cleaned text"* more robust than *"take characters 0 to 5 of the second cell"*?

*Your answer:*

## B5. Many items: one book first, then all of them

Open https://books.toscrape.com/catalogue/page-1.html and **Inspect** one book. Each is an `<article class="product_pod">`. Inside it:

| What | Where | CSS selector |
|---|---|---|
| Full title | the `title` **attribute** of the link in the `<h3>` | `h3 a` |
| Price | text of `<p class="price_color">` | `p.price_color` |
| Rating | the second **class name** of `<p class="star-rating Three">` | `p.star-rating` |
| Link to the book's page | the `href` attribute of the same link | `h3 a` |

> **CSS selectors in one line each:** `tag` matches a tag; `.name` matches a class; `tag.name` both; `parent child` an element inside another; `#id` an id.

First fetch the page and look at **one** book.

In [ ]:
# 🟢 Fetch the first page (robots.txt checked first, as always)
BOOKS = "https://books.toscrape.com/catalogue/page-1.html"
books_rp = RobotFileParser("https://books.toscrape.com/robots.txt")
try:
    books_rp.read()
    books_allowed = books_rp.can_fetch(USER_AGENT, BOOKS)
except Exception:
    books_allowed = None
print("Allowed by robots.txt:", books_allowed, "(a missing robots.txt counts as allowed)")

SAMPLE_BOOKS = "".join(
    f'<article class="product_pod"><p class="star-rating {r}"></p><h3><a href="book-{i}_1/index.html" '
    f'title="Sample book {i}">Sample book {i}...</a></h3><div class="product_price">'
    f'<p class="price_color">£{10 + i}.99</p><p class="instock availability">In stock</p></div></article>'
    for i, r in enumerate(["One", "Two", "Three", "Four", "Five"] * 4)) + '<li class="next"><a href="page-2.html">next</a></li>'

page_html, src = fetch_html(BOOKS, allowed_by_robots=books_allowed)
if page_html is None:
    page_html, src = SAMPLE_BOOKS, "sample"
books_soup = BeautifulSoup(page_html, "html.parser")
pods = books_soup.select("article.product_pod")
print("Using:", src, "|", len(pods), "books on the page")

first = pods[0]
print(first.prettify()[:700])

**What you should see:** **20 books** on the page, and the HTML of the first one, with the `title`, `href` and `class` attributes visible.

🟡 Extract the four fields from **one** book, using the helpers from B2 and the table above.

In [ ]:
title = attr_or_none(first, "h3 a", "____")
price_text = text_or_none(first, "____")
rating_classes = attr_or_none(first, "p.star-rating", "____")      # a list, such as ['star-rating', 'Three']
link = urljoin(BOOKS, attr_or_none(first, "h3 a", "____") or "")

if filled(title, price_text, rating_classes, link):
    print(title, "|", price_text, "|", rating_classes, "|", link)
    check(title and not title.endswith("...") and price_text.startswith("£") and "star-rating" in rating_classes,
          "title attribute; p.price_color; class attribute; href attribute.")

✍️ **Analyse.** The visible link text ends in `...`, but the `title` attribute does not. Why might a website keep the full value in an attribute? What does that tell you about where to look for clean data?

*Your answer:*

Now turn the one-book code into a function, and run it on **every** book.

🔵 Complete `parse_book(pod)` so it returns a dictionary with `title`, `price_gbp` (a **number**), `rating` (a **number** from 1 to 5) and `url`. Hints: remove everything except digits and the decimal point with `re.sub(r"[^0-9.]", "", text)`; map the rating word with the `RATINGS` dictionary.

In [ ]:
RATINGS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_book(pod):
    # Your code here
    pass

books = pd.DataFrame([parse_book(p) for p in pods])
books.head()

In [ ]:
# Test for B5
check(len(books) == len(pods) and list(books.columns) == ["title", "price_gbp", "rating", "url"], "Return the four keys for every book.")
check(books["price_gbp"].dtype == float and books["rating"].between(1, 5).all(), "price_gbp as a float; rating as a number 1 to 5.")

## B6. Many pages: follow the "next" link

At the bottom of each page is a **next** link, `<li class="next"><a href="page-2.html">`. Follow the site's own links rather than inventing page addresses, and pause between requests. (`fetch_html` already pauses one second after each live request.)

🟡 Complete the loop to collect **three pages**.

In [ ]:
all_books, url = [], BOOKS
for _ in range(3):                                      # only what we need
    page_html, src = fetch_html(url, allowed_by_robots=books_allowed)
    if page_html is None:
        break
    page_soup = BeautifulSoup(page_html, "html.parser")
    all_books.extend(parse_book(p) for p in page_soup.select("____"))
    next_link = attr_or_none(page_soup, "li.next a", "____")
    if next_link is None:
        break
    url = urljoin(url, ____)

books3 = pd.DataFrame(all_books)
print(books3.shape)

In [ ]:
# Test for B6 (needs a connection: three live pages)
if books3.shape[0] >= 60:
    check(books3["title"].is_unique, "Each page should add new books: are you updating url?")
    check(books3.shape[0] == 60, "Three pages of 20 books.")
else:
    print("Fewer than 60 books: you may be offline (then this check is skipped), or the loop stops early.")

✍️ **Reflect.** The site has 50 pages. Why does the loop stop at three? Write one sentence on how you would decide how many pages a project actually needs.

*Your answer:*

## B7. Two pages that look the same

Open https://quotes.toscrape.com/ and https://quotes.toscrape.com/js/ in two tabs. They look identical. ✍️ Predict how many `div.quote` elements Python will find on each: ____ and ____

In [ ]:
# 🟢 Run after predicting
for url in ["https://quotes.toscrape.com/", "https://quotes.toscrape.com/js/"]:
    page, src = fetch_html(url)
    if page:
        n = len(BeautifulSoup(page, "html.parser").select("div.quote"))
        print(f"{n:>3} div.quote elements on {url}\n")

**Key output:** 10 on the first page and **0** on the second. The second page sends almost no HTML, plus a **JavaScript** program that builds the quotes inside your browser. `requests` downloads the page but does not run JavaScript.

What to do, in order of preference:

1. Open the browser's developer tools, **Network** tab, reload, and look for the request that carries the data, often JSON. That is an API in disguise; call it directly.
2. Look inside the page's `<script>` elements: sometimes the data is there.
3. Only then consider browser automation (Playwright or Selenium), which is slower and heavier.

🟡 Find the data inside the script.

In [ ]:
page, src = fetch_html("https://quotes.toscrape.com/js/")
if page:
    js_soup = BeautifulSoup(page, "html.parser")
    scripts = [s.get_text() for s in js_soup.find_all("script") if "var data" in s.get_text()]
    print(scripts[0][:300] if scripts else "No script containing data")
    if scripts:
        match = re.search(r"var data = (\[.*?\]);", scripts[0], re.S)
        quotes_js = json.loads(match.group(1))
        quotes = pd.json_normalize(quotes_js, sep="____")
        print(quotes[["author_name", "text"]].head(3))
        check(len(quotes) == 10, "sep='_' gives the column author_name.")

**Key output:** the script contains a JavaScript list of quote objects that happens to be valid JSON, so `json.loads` reads it and `json_normalize` (from the API lab) flattens it. The lesson: before scraping a JavaScript page, look for the data's real source.

---
# Part C. Research exercises

## C1. Case study: 70,000 dating-site profiles

### What happened

In May 2016, two Danish researchers, Emil Kirkegaard and Julius Daugbjerg Bjerrekær, released a dataset on the **Open Science Framework** (OSF), a repository for sharing research data, containing details of almost **70,000 users** of the dating site **OkCupid**. Read the sources below before you write anything.

From press reports at the time:

* They wrote software to scrape the profiles, collecting **usernames** (not real names), age, gender, location, religion and personality traits, and the users' answers to the questions the site uses to suggest matches. They did not ask OkCupid or its users for permission.
* The data were collected between November 2014 and March 2015, from profiles that had answered many questions, including the answers to about 2,600 of the most popular questions. Some answers concerned drug use, sexual preferences, political leanings and attitudes to homosexuality.
* The researchers argued the data were already public. Their draft paper said that all the data were, or had been, publicly available, so releasing them merely presented them in a more useful form.
* Critics noted that usernames combined with location and detailed answers could allow people to be re-identified. OkCupid users can also restrict their profiles to logged-in members, so profiles not meant to be publicly viewable may have been collected.
* OkCupid called the posting a clear violation of its terms of service. The OSF removed the dataset after a copyright (DMCA) claim from OkCupid, but it had already been downloaded **more than 500 times**.
* Aarhus University distanced itself, stating that the project was not part of the student's work at the university.

A useful contrast: an earlier statistics-education paper (Kim and Escobedo-Land, 2015) used OkCupid profile data **with explicit permission from OkCupid's chief executive**, and **removed usernames** as a basic step to de-identify users.

**Sources to read:**

* Vox (Brian Resnick, 12 May 2016): https://www.vox.com/2016/5/12/11666116/70000-okcupid-users-data-release
* Summary by Privacy International: https://privacyinternational.org/examples/1857/researchers-release-dataset-70000-users-okcupid-profiles-and-call-data-public
* Engadget on the takedown: https://engadget.com/2016/05/17/publicly-released-okcupid-profiles-taken-down-dmca-claim
* Baumer et al., *Integrating data science ethics into an undergraduate major* (discusses three uses of OkCupid data): https://arxiv.org/abs/2001.07649

### Task 1: which checks were skipped?

✍️ For each check below, write whether it was done, skipped, or unclear from the sources, and give the evidence in one sentence. Use the checks from the lecture and from Part A.

| Check | Done, skipped or unclear? | Evidence |
|---|---|---|
| Is there an API, a licensed dataset, or permission from the site? | | |
| Terms of service read and respected | | |
| robots.txt and access controls (logged-in-only profiles) respected | | |
| Personal data identified, including special category data (sex life, sexual orientation, political opinions, religious beliefs, health) | | |
| Lawful basis and ethics review before collection | | |
| Data minimisation: only what the research question needs | | |
| De-identification before release (usernames, location, rare answers) | | |
| Risk of harm to the people in the data assessed | | |
| Licence and rights to republish others' content considered | | |
| Secure storage and controlled access, rather than an open public release | | |

### Task 2: "the data were public"

✍️ In one paragraph (about 150 words), evaluate the researchers' argument that publishing the dataset was acceptable because the data were already public. Use at least two of: UK GDPR's treatment of publicly available personal data; special category data; re-identification through combining fields; the terms of service; the difference between *viewing* data and *republishing* it at scale.

*Your answer:*

### Task 3: a defensible collection plan

✍️ Suppose the research question were legitimate: *"Do stated political views predict the answers people give to compatibility questions?"* Draft the collection plan that would make the work defensible. Keep each answer to one or two sentences.

| Item | Your plan |
|---|---|
| Research question and why this data is needed | |
| Route to the data (API, data-sharing agreement, permission, or a different source) | |
| Terms of service and robots.txt: what they say and how you comply | |
| Ethics review: which body, and before what point | |
| Lawful basis under UK GDPR, and the condition for special category data | |
| Minimum fields needed (list them), and fields you will **not** collect | |
| De-identification: what you remove, aggregate or coarsen (e.g. exact location) | |
| Storage: where, who can access it, encryption | |
| Retention: how long, then deletion | |
| What you will publish: aggregated results only, or a dataset? Under what access controls? | |
| Provenance record: what you log for each collection run | |
| How a participant could object or ask for removal | |

### Task 4: code you would not write

🔵 Without touching any real site, write a function `minimise(record)` that takes one invented profile dictionary and returns only the fields your plan allows, with location coarsened to the country and age turned into a ten-year band. Test it on the invented example.

In [ ]:
example = {"username": "sunny_dev_88", "age": 34, "city": "Manchester", "country": "United Kingdom",
           "religion": "none", "politics": "liberal", "answers": {"q101": "yes", "q205": "no"}}

def minimise(record):
    # Your code here
    pass

print(minimise(example))

In [ ]:
# Test for C1 Task 4
m = minimise(example)
check("username" not in m and "city" not in m, "Drop identifiers and fine-grained location.")
check(m.get("age_band") == "30-39", "Turn 34 into the band '30-39'.")

✍️ **Reflect.** Even after `minimise`, could someone be re-identified? Think about rare combinations of answers. What else would you do before sharing any data?

*Your answer:*

## C2. A pre-collection check for your project

Choose a website you might scrape for your project. Run the checks **before** collecting anything.

🔵 Complete `preflight(url)`, which returns a dictionary with: the robots.txt verdict for our user agent; the page's status code and content type (one polite request); and whether the page mentions an API or a download (search the text for the words `api`, `download` or `csv`).

In [ ]:
def preflight(url):
    # Your code here
    pass

project_url = "https://books.toscrape.com/"          # replace with your candidate site
preflight(project_url)

✍️ **Go or no go?** Then answer, for your candidate site:

| Question | Your answer |
|---|---|
| Is there an API or a download that makes scraping unnecessary? | |
| What do robots.txt and the terms of use allow? Quote one line of the terms. | |
| Does the data contain personal data? Special category data? | |
| What is the minimum you need, in pages and in fields? | |
| Decision: scrape, use another route, or choose another source? Why? | |

## C3. Collect once, parse many times

Professional scrapers keep **collection** and **parsing** separate: download each page once, save it as raw HTML, then parse the saved files as often as needed. If you fix a bug in the parser, you re-parse; you do not download again.

🔵 1. Write `collect_pages(start_url, n_pages)` that follows the next links from books.toscrape.com, saves each page's raw HTML in `data/raw/books/` with the time it was collected, and appends one provenance line per page (URL, file name, SHA-256 hash, status). 2. Write `parse_saved(folder)` that reads **only** the saved files and returns one DataFrame of books, using `parse_book`. 3. Show that running `parse_saved` twice gives identical results without any network request.

In [ ]:
# Your code here

---
## Before you leave

- [ ] Part A runs, and you can describe the six steps of the scraping routine, starting with *check whether you may scrape*.
- [ ] You can explain each cleaning step in Example 2, and why a pattern beats fixed character positions.
- [ ] Every Part B test shows ✅ (or is skipped because you are offline).
- [ ] You have started the C1 case study tables.
- [ ] **Edit → Clear all outputs**, download the notebook into your personal repository and commit it. Keep `data/raw/` out of Git; commit `data/provenance.jsonl`.